# Day 2 — Extra exercises: updating and one-parameter models

**For two kinds of reader.** If you **missed Day 2** (after informing the lecturer in advance), this notebook is part of your make-up work. If you attended and want **more practice**, work through any part at your own pace; nothing here is required for you.

**What a make-up submission for Day 2 contains** (see `../day-2-makeup.md`):

- **A1–A3 and B1–B3** of this notebook;
- your own solutions to the core in-class exercises in `../day-2-exercises-morning.ipynb` and `../day-2-exercises-afternoon.ipynb`.

The reflection prompts are optional. **C-problems are optional challenges** for everyone. The make-up submission is graded on the 0–4 in-class scale and counts as your in-class score for the missed day.

**Topics.** Part A: beta–binomial updating, prior sensitivity, sequential updating and prediction (morning lecture). Part B: gamma–Poisson updating with exposure, comparing two rates, predicting a count and choosing a prior from expert knowledge (afternoon lecture). About 3–4 hours for A and B together.

**Background.** Go through the Day 2 slides first (`../day-2-slides-morning.pptx` and `../day-2-slides-afternoon.pptx`); demos 1 and 3 in `../demos/` work through the same ideas step by step. Each task below names the slide it builds on.

**How to work.** Open in JupyterLab, select **R (bayes)**, run the cells in order from a fresh kernel. Hints are hidden under **Hint**; try first. Write explanations in your own words in the *Your answer* cells: numbers alone are not enough. All data are **synthetic teaching data**; never report them as real measurements. No Stan is needed.

**Submitting (make-up).** Submit this notebook with outputs and your written answers, together with the in-class exercises, through the announced route by the announced deadline (normally within one week of the missed day). Discussing ideas is fine; the code and explanations must be your own. You may use AI assistants without declaring it, but you should be able to explain every line.

**R used here.** `tibble()` makes a small table; `|>` reads as "and then"; `mutate()` adds a column; `summarise()` collapses rows to one summary row; `group_by()` makes `summarise()` work per group; `bind_rows()` stacks tables; `count(x)` counts how often each value of `x` occurs. Distributions use the four letters from the slides: `d` (probability or density), `p` (P(X ≤ x)), `q` (quantile), `r` (random draws).

## Setup

Run this cell first.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # tibble, dplyr and ggplot2
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")

## Part A — Germination of a new seed lot

A plant-breeding group tests a new seed lot. In a first batch, **18 of 25** seeds germinated. We want to learn **θ, the probability that one seed of this lot germinates**.

- **Likelihood:** seeds germinate independently, all with the same θ, so the number germinating is y | θ ~ Binomial(25, θ).
- **Prior:** the supplier's previous lots germinated at "about two thirds". The group encodes this as **Beta(4, 2)**: mean 4 / (4 + 2) = 0.67, but weak, worth about 4 + 2 = 6 seeds.

### A1. One posterior, by hand and in R (25 min)

*Background: slides "The beta–binomial update" and "Summarising the posterior Beta(16, 8)" (morning).*

(a) Write the likelihood as a function of θ (up to a constant) and show, by multiplying it with the prior, that the posterior is Beta(4 + 18, 2 + 7).
(b) Compute the posterior mean a / (a + b), a 90% interval `qbeta(c(0.05, 0.95), a, b)` and P(θ > 0.6 | data) = `1 - pbeta(0.6, a, b)`.
(c) Interpret each of the three numbers in one sentence for a plant breeder.

In [ ]:
n1 <- 25; y1 <- 18     # first batch: 18 of 25 germinated
a0 <- 4;  b0 <- 2      # prior Beta(4, 2)

In [ ]:
# a1 <- a0 + ...;  b1 <- b0 + ...
# a1 / (a1 + b1);  qbeta(c(0.05, 0.95), a1, b1);  1 - pbeta(0.6, a1, b1)

<details><summary><b>Hint</b> (click to open)</summary>

Likelihood ∝ θ¹⁸(1 − θ)⁷. Prior ∝ θ^(4−1)(1 − θ)^(2−1). Multiply and add the exponents of θ and of (1 − θ) separately.

</details>

**Your answer:** *(write here)*

### A2. How much does the prior matter? (25 min)

*Background: slide "How much does the prior matter?" (morning) and the prior simulation in the morning exercise A2.*

Repeat A1 with three priors:

| prior | a | b | meaning |
|---|---|---|---|
| uniform | 1 | 1 | no preference |
| supplier | 4 | 2 | "about two thirds", weak |
| sceptical | 30 | 30 | a colleague who insists θ is close to 0.5, strong (worth 60 seeds) |

(a) Make a table with each posterior's mean, 90% interval and P(θ > 0.6).
(b) Plot the three posteriors in one figure, with a vertical line at the observed proportion 18/25.
(c) **Before** looking at the data, what does each prior expect to see? Simulate the number of germinating seeds out of 25 under each prior, in two steps: draw θ from the prior, then draw a binomial count for that θ, `rbinom(5000, 25, rbeta(5000, a, b))`. Compare the middle 90% (5% and 95% quantiles) of the simulated counts. Which prior makes a strong claim, and what evidence would justify it?

In [ ]:
priors <- tibble(
  prior = c("uniform", "supplier", "sceptical"),
  a     = c(1, 4, 30),
  b     = c(1, 2, 30)
)
priors

In [ ]:
# (a) priors |> mutate(a_post = a + y1, b_post = b + n1 - y1, mean = ..., lower_90 = ..., upper_90 = ..., p_above_6 = ...)
# (b) theta <- seq(0.001, 0.999, length.out = 500)
#     posteriors <- bind_rows(
#       tibble(theta = theta, density = dbeta(theta, 1 + y1, 1 + n1 - y1), prior = "uniform"),
#       ...
#     )
#     ggplot(posteriors, aes(x = theta, y = density, colour = prior)) + geom_line() + geom_vline(xintercept = y1 / n1)
# (c) counts_uniform <- rbinom(5000, 25, rbeta(5000, 1, 1));  quantile(counts_uniform, c(0.05, 0.95))

<details><summary><b>Hint</b> (click to open)</summary>

For (a), each row of `priors` is one prior; `mutate()` computes the posterior for every row at once.

</details>

**Your answer:** *(write here)*

### A3. A second batch and a prediction (25 min)

*Background: slides "Updating one departure at a time" (morning), "Posterior prediction: two kinds of uncertainty" and "Predicting the next 10 departures" (afternoon).*

A second batch of **40** seeds gives **27** germinations.

(a) Update the A1 posterior Beta(22, 9) with the second batch only (the A1 posterior is now the prior). Check that you get the same posterior by updating Beta(4, 2) once with all 65 seeds.
(b) The group will plant **30** seeds next week. Using the final posterior, simulate how many will germinate, in two steps: draw θ from the posterior, then a count of 30 seeds for that θ. Report the mean, a 90% predictive interval and P(at least 20 germinate).
(c) Compare with a **plug-in** prediction that fixes θ at the posterior mean: `rbinom(10000, 30, posterior mean)`. Which is wider, and why?

In [ ]:
n2 <- 40; y2 <- 27     # second batch: 27 of 40 germinated

In [ ]:
# (a) a2 <- a1 + y2;  b2 <- b1 + n2 - y2;  compare with a0 + y1 + y2 and b0 + (n1 - y1) + (n2 - y2)
# (b) theta_draw <- rbeta(10000, a2, b2);  y_new <- rbinom(10000, 30, theta_draw)
#     mean(y_new);  quantile(y_new, c(0.05, 0.95));  mean(y_new >= 20)
# (c) y_plug <- rbinom(10000, 30, a2 / (a2 + b2));  compare sd(y_new) with sd(y_plug)

<details><summary><b>Hint</b> (click to open)</summary>

Sequential updating keeps adding successes to a and failures to b. For (c), ask which of the two kinds of uncertainty the plug-in leaves out.

</details>

**Your answer:** *(write here)*

### Reflection A (optional)

In 80–120 words, explain to a plant breeder the difference between the 90% interval for θ (A1) and the 90% predictive interval for next week's count (A3). Which one answers "how many seedlings should I expect?".

**Your answer:** *(write here)*

## Part B — Bird counts with unequal observation time

Volunteers count a ground-nesting bird at two sites. Each watch has a different **duration**:

| site | counts (birds seen) | hours watched (exposure) |
|---|---|---|
| 1 | 3, 7, 2, 5, 9 | 0.5, 1.5, 0.5, 1, 2 |
| 2 | 1, 4, 0, 2, 3, 1 | 1, 1.5, 0.5, 1, 2, 1 |

- **Count** cᵢ: the number of birds seen in watch i (0 is a real observation).
- **Exposure** tᵢ: how long watch i lasted, in hours.
- **Rate** λ: the expected number of birds **per hour** at a site. Watch i is expected to see λ × tᵢ birds.

**Model** (*slides "Exposure: counts over different lengths of time" and "The gamma–Poisson update", afternoon*): cᵢ | λ ~ Poisson(λ tᵢ), watches independent, the same λ for all watches at a site. Prior for each site's λ: **Gamma(shape 2, rate 1)**, mean 2 birds per hour.

### B1. Two posterior rates (25 min)

(a) Show that the posterior for a site is Gamma(2 + Σcᵢ, 1 + Σtᵢ): add the total count to the shape and the total hours to the rate.
(b) Compute each site's posterior mean (shape / rate), 90% interval and P(λ > 3 per hour). In R always name the arguments: `qgamma(0.05, shape = ..., rate = ...)`.
(c) Why would it be wrong to average the raw counts per watch, ignoring the hours?

In [ ]:
c1 <- c(3, 7, 2, 5, 9);    t1 <- c(0.5, 1.5, 0.5, 1, 2)      # site 1: counts and hours
c2 <- c(1, 4, 0, 2, 3, 1); t2 <- c(1, 1.5, 0.5, 1, 2, 1)     # site 2
a0 <- 2; b0 <- 1                                              # prior Gamma(shape = 2, rate = 1)

In [ ]:
# shape1 <- a0 + sum(c1);  rate1 <- b0 + sum(t1)    and the same for site 2
# shape1 / rate1;  qgamma(c(0.05, 0.95), shape = shape1, rate = rate1);  1 - pgamma(3, shape = shape1, rate = rate1)

<details><summary><b>Hint</b> (click to open)</summary>

Poisson likelihood with exposure: ∏ (λtᵢ)^cᵢ e^(−λtᵢ) ∝ λ^(Σcᵢ) e^(−λ Σtᵢ). The Gamma(a, b) prior is ∝ λ^(a−1) e^(−bλ). Multiply and collect the powers of λ and the terms in the exponent.

</details>

**Your answer:** *(write here)*

### B2. Comparing sites and predicting a count (25 min)

*Background: slides "Running it from R" (morning: a posterior probability is the share of draws for which a statement is true), "A rate versus a future count" and "When the parameter itself is uncertain" (afternoon).*

Formulas give us each site's posterior, but not directly an answer to "is site 1 busier than site 2?". **Draws** make this easy: draw many rates for each site from its posterior, and look at them draw by draw.

(a) Draw 5,000 rates for each site with `rgamma()`. The two sites are independent, so draw them separately. Estimate P(λ₁ > λ₂) as the share of draws where site 1's rate is larger, `mean(l1 > l2)`, and a 90% interval for the ratio λ₁ / λ₂ from the quantiles of `l1 / l2`.
(b) A volunteer will watch **site 2 for 2 hours**. Simulate the count in two steps: for each draw of λ₂, draw a Poisson count with mean λ₂ × 2. Report the mean, a 90% predictive interval and P(count ≥ 6).
(c) Check P(count ≥ 6) exactly. The future count is **negative binomial**; in R, with the posterior Gamma(a, b) and exposure t: `1 - pnbinom(5, size = a, prob = b / (b + t))`. Compare also with a Poisson that fixes λ₂ at its posterior mean, `1 - ppois(5, 2 * a / b)`.

In [ ]:
set.seed(2026)
# (a) l1 <- rgamma(5000, shape = shape1, rate = rate1);  l2 <- ...
#     mean(l1 > l2);  quantile(l1 / l2, c(0.05, 0.95))
# (b) y_new <- rpois(5000, lambda = l2 * 2)
# (c) 1 - pnbinom(5, size = shape2, prob = rate2 / (rate2 + 2))

<details><summary><b>Hint</b> (click to open)</summary>

Every pair of draws (l1[i], l2[i]) is one plausible pair of rates. `l1 > l2` is TRUE or FALSE for each pair, so `mean(l1 > l2)` is the share of pairs where site 1 is busier.

</details>

**Your answer:** *(write here)*

### B3. A prior from expert knowledge (25 min)

*Background: the afternoon exercise B3 (a prior predictive check).*

An ornithologist says: *"At sites like these, I'd be 90% sure the rate is between 1 and 10 birds per hour."* In other words, the 5% and 95% quantiles of the prior should be about 1 and 10.

(a) Find a Gamma(shape, rate) prior whose 5% and 95% quantiles are close to 1 and 10, by trial and error: try a few values with `qgamma(c(0.05, 0.95), shape = ..., rate = ...)` and adjust. (Tip: a smaller rate stretches the distribution to larger values; a larger shape makes it narrower relative to its mean.)
(b) What does each prior expect to see in a **1-hour watch** before any data? Simulate counts in two steps, `rpois(5000, rgamma(5000, shape, rate))`, under the original Gamma(2, 1) and under your elicited prior, and compare the 5%, 50% and 95% quantiles. Which matches the ornithologist's experience?
(c) Recompute site 2's posterior mean and 90% interval under the elicited prior and compare with B1. Why does the prior matter more for site 2 than for site 1?

In [ ]:
# (a) qgamma(c(0.05, 0.95), shape = 2, rate = 0.5)   # try, then adjust shape and rate
# (b) counts_old <- rpois(5000, rgamma(5000, shape = 2, rate = 1));  quantile(counts_old, c(0.05, 0.5, 0.95))
# (c) shape2_el <- your_shape + sum(c2);  rate2_el <- your_rate + sum(t2)

<details><summary><b>Hint</b> (click to open)</summary>

The prior Gamma(a, b) acts like a birds seen in b hours. Compare that with each site's totals (site 1: 26 birds in 5.5 hours; site 2: 11 birds in 7 hours).

</details>

**Your answer:** *(write here)*

### Reflection B (optional)

In 80–120 words: why is the uncertainty about the next watch's **count** larger than the uncertainty about the **rate**? Name one assumption of the Poisson model that you would check with more data (for example time of day, observer, or birds arriving in groups).

**Your answer:** *(write here)*

## C. Optional challenges

**C1. The posterior mean is a compromise** (*slide "The posterior mean is a compromise"*). Show algebraically that the beta–binomial posterior mean (a + y) / (a + b + n) equals a weighted average of the prior mean a / (a + b) and the sample proportion y / n, with weight n / (a + b + n) on the data. Check it numerically for the three priors in A2.

**C2. Base rates** (*Day 1, the marble game: Bayes' rule with two hypotheses*). A test for a seed-borne pathogen detects an infected lot with probability 0.95 and wrongly flags a clean lot with probability 0.03; 2% of lots are infected. A lot tests positive. What is P(infected | positive)? The lot is retested independently and is positive again. Update.

**C3. Which prior is "flat"?** (*afternoon exercise, optional extension*) A uniform prior for λ on (0, 20) is not uniform for log λ. Simulate 10,000 values of λ from Uniform(0, 20) and plot a histogram of log(λ).

In [ ]:
# Optional challenge work

**Your answer:** *(write here)*

## Make-up checklist

- [ ] A1–A3 and B1–B3 · [ ] in-class exercises (`../day-2-exercises-morning.ipynb` and `../day-2-exercises-afternoon.ipynb`)
- [ ] The derivations in A1(a) and B1(a) are written out, not only computed
- [ ] The notebook runs from a fresh kernel; every answer cell has your own explanation
- [ ] I can explain every line of my code